In [ ]:
import gc
import os
import json
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import wandb

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_evaluation.ipynb"
wandb.login()
compute_dtype = torch.bfloat16
model_id = "mistralai/Voxtral-Mini-3B-2507"
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_path = "./models/voxtral-glados-sft/final_adapters"

processor = AutoProcessor.from_pretrained(model_id)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)
base_model = VoxtralForConditionalGeneration.from_pretrained(
        model_id,
        attn_implementation="flash_attention_2",
        device_map="auto",
        low_cpu_mem_usage=True,
        quantization_config=bnb_config,
        dtype=compute_dtype
    )
peft_config = PeftConfig.from_pretrained(lora_path)
peft_config.init_lora_weights = False # avoid crash due to quantized initialization
model = PeftModel.from_pretrained(
    base_model,
    lora_path,
    config=peft_config,
    is_trainable=False
)
model.eval()
wandb.init(
    project=os.environ["WANDB_PROJECT"],
    job_type="evaluation",
    name="voxtral-glados-test-eval"
)

In [ ]:
def extract_json_payloads(text):
    if not isinstance(text, str):
        return []
    payloads = []
    # Find all non-overlapping { ... } blocks
    matches = re.finditer(r'(\{.*?\})', text, re.DOTALL)
    for match in matches:
        try:
            payloads.append(json.loads(match.group(1)))
        except json.JSONDecodeError:
            pass
    return payloads

def parse_generated_text(text):
    parts = text.split('\n\n', 1)
    json_section = parts[0]
    text_response = parts[1] if len(parts) > 1 else ""
    pred_payloads = extract_json_payloads(json_section)
    return pred_payloads, text_response, json_section

def calculate_slot_f1_multi(true_payloads, pred_payloads):
    def get_slot_set(payloads):
        slots = set()
        for p in payloads:
            if not p: # Handle empty dictionary {}
                continue
            # Associate the slot with its specific service to prevent cross-intent collisions
            srv = p.get("service", "unknown")
            for k, v in p.items():
                if k != "service":
                    slots.add((srv, k, str(v)))
        return slots
    true_set = get_slot_set(true_payloads)
    pred_set = get_slot_set(pred_payloads)
    # If both true and predicted require NO slots (Information Request {}), it's a perfect match.
    if not true_set and not pred_set:
        return 1.0
    tp = len(true_set.intersection(pred_set))
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    if (tp + fp) == 0 or (tp + fn) == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if (precision + recall) == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

#### Load test dataset and evaluate

In [ ]:
# Load your test dataset (using a CSV structure similar to your training script)
test_df = pd.read_csv("data/combined_multimodal_dataset_test.csv")
eval_table = wandb.Table(columns=[
    "Audio File",
    "True Payload",
    "True Response",
    "Predicted Payload",
    "Predicted Response"
])
results = []
# Define the exact text prompt used during training
ha_data = "Services: climate.set_fan_mode(fan_mode), climate.set_humidity(humidity), climate.set_hvac_mode(), climate.set_preset_mode(), climate.set_temperature(temperature), climate.toggle(), climate.turn_off(), climate.turn_on(), cover.close_cover(), cover.open_cover(), cover.stop_cover(), cover.toggle(), fan.decrease_speed(), fan.increase_speed(), fan.toggle(), fan.turn_off(), fan.turn_on(), light.toggle(), light.turn_off(), light.turn_on(rgb_color,brightness), lock.lock(), lock.unlock(), media_player.media_next_track(), media_player.media_pause(), media_player.media_play(), media_player.media_play_pause(), media_player.media_previous_track(), media_player.media_stop(), media_player.toggle(), media_player.turn_off(), media_player.turn_on(), media_player.volume_down(), media_player.volume_mute(), media_player.volume_up(), switch.toggle(), switch.turn_off(), switch.turn_on(), timer.add_item(item), timer.cancel(), timer.pause(), timer.start(duration), vacuum.pause(), vacuum.return_to_base(), vacuum.start(), vacuum.stop()\n"
SYSTEM_INSTRUCTION = f"You are GLaDOS, an AI assistant that controls the devices in a house. Execute the spoken command, output the required JSON payload, and respond in character. Complete the following task as instructed or answer the following question with the information provided only.\n{ha_data}"
BATCH_SIZE = 8
total_processed = 0
successful_parses = 0
correct_intents = 0
cumulative_slot_f1 = 0
strict_format_adherence = 0
for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Evaluating Test Set (Batched)"):
    batch_df = test_df.iloc[i:i+BATCH_SIZE]
    conversations = []
    valid_rows = []
    # Prepare the batch conversations
    for idx, row in batch_df.iterrows():
        audio_path = os.path.join("data/synthesized_train_16k/", row["Audio_File"])
        if not os.path.exists(audio_path):
            continue
        conversations.append([
            {"role": "user", "content": [
                {"type": "text", "text": SYSTEM_INSTRUCTION},
                {"type": "audio", "path": audio_path}
            ]}
        ])
        valid_rows.append(row)
    if not conversations:
        continue
    # Process Audio and Text
    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True}
    ).to(model.device, dtype=torch.bfloat16)
    # Generate Output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )
    # Extract only the newly generated tokens
    input_len = inputs.input_ids.shape[1]
    generated_texts = processor.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    # Parse and record results for each item in the batch
    for row, gen_text in zip(valid_rows, generated_texts):
        true_payloads = extract_json_payloads(row["Assistant_Payload"])
        pred_payloads, pred_text_response, raw_json_section = parse_generated_text(gen_text)
        strict_format_passed = gen_text.lstrip().startswith("{")
        results.append({
            "audio_file": row["Audio_File"],
            "true_payload": true_payloads,
            "true_response": row["Target_GLaDOS_Response"],
            "predicted_payload": pred_payloads,
            "predicted_response": pred_text_response
        })
        eval_table.add_data(
            row["Audio_File"],
            str(true_payloads),
            row["Target_GLaDOS_Response"],
            str(pred_payloads),
            pred_text_response
        )
        total_processed += 1
        if len(pred_payloads) > 0:
            successful_parses += 1
        true_intents = {p.get("service") for p in true_payloads}
        pred_intents = {p.get("service") for p in pred_payloads}
        if true_intents == pred_intents:
            correct_intents += 1
        cumulative_slot_f1 += calculate_slot_f1_multi(true_payloads, pred_payloads)
        if strict_format_passed:
            strict_format_adherence += 1
    running_json_rate = (successful_parses / total_processed) * 100
    running_intent_acc = (correct_intents / total_processed) * 100
    running_slot_f1 = (cumulative_slot_f1 / total_processed) * 100
    running_strict_acc = (strict_format_adherence / total_processed) * 100
    running_slu_f1 = 0.0
    if (running_intent_acc + running_slot_f1) > 0:
        running_slu_f1 = (2 * (running_intent_acc/100) * (running_slot_f1/100)) / ((running_intent_acc/100) + (running_slot_f1/100)) * 100
    wandb.log({
        "live_metrics/json_parse_rate": running_json_rate,
        "live_metrics/intent_accuracy": running_intent_acc,
        "live_metrics/slot_f1_score": running_slot_f1,
        "live_metrics/ifeval_strict_accuracy": running_strict_acc,
        "live_metrics/slu_f1_score": running_slu_f1,
        "live_metrics/samples_processed": total_processed
    })
    # Save a local CSV backup every 50 batches (skipping the 0th batch)
    if i > 0 and (i // BATCH_SIZE) % 50 == 0:
        # Convert our current results list to a DataFrame and save it safely
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv("data/evaluation_checkpoint_latest.csv", index=False)
        print(f"\n[Checkpoint] Saved backup of {len(results)} rows to disk.")

#### Compute Metrics

In [ ]:
total_samples = len(results)
successful_parses = 0
correct_intents = 0
slot_f1_scores = []
for res in results:
    true_p = res["true_payload"]
    pred_p = res["predicted_payload"]
    # Syntactic Validity (Did it output at least one JSON bracket structure?)
    if len(pred_p) > 0:
        successful_parses += 1
    # Intent Accuracy (Evaluated as a set to handle multiple actions)
    # E.g., ["lock.lock", "cover.close"] vs ["lock.lock", "cover.close"]
    # An empty payload {} will yield [None]
    true_intents = {p.get("service") for p in true_p}
    pred_intents = {p.get("service") for p in pred_p}
    if true_intents == pred_intents:
        correct_intents += 1
    # Slot F1-Score (Calculates across all JSONs; handles empty {} perfectly)
    slot_f1 = calculate_slot_f1_multi(true_p, pred_p)
    slot_f1_scores.append(slot_f1)
# Final Computations
json_parse_rate = (successful_parses / total_samples) * 100 if total_samples > 0 else 0
intent_accuracy = (correct_intents / total_samples) * 100 if total_samples > 0 else 0
avg_slot_f1 = (sum(slot_f1_scores) / total_samples) * 100 if total_samples > 0 else 0
print("=== Phase 3: Evaluation Metrics ===")
print(f"Total Test Samples: {total_samples}")
print(f"JSON Parse Rate (Syntactic Validity): {json_parse_rate:.2f}%")
print(f"Intent Accuracy (Multi-Intent / Info Request): {intent_accuracy:.2f}%")
print(f"Slot F1-Score (Semantic Accuracy): {avg_slot_f1:.2f}%")
if (intent_accuracy + avg_slot_f1) > 0:
    slu_f1 = (2 * (intent_accuracy/100) * (avg_slot_f1/100)) / ((intent_accuracy/100) + (avg_slot_f1/100)) * 100
else:
    slu_f1 = 0.0
print(f"SLU-F1 (Unified semantic comprehension): {slu_f1:.2f}%")
# Push metrics to W&B
wandb.log({
    "metrics/json_parse_rate": json_parse_rate,
    "metrics/intent_accuracy": intent_accuracy,
    "metrics/slot_f1_score": avg_slot_f1,
    "metrics/slu_f1_score": slu_f1
})
# Push the interactive table
wandb.log({"evaluation_results_table": eval_table})
wandb.finish()